In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("../data/fraudTrain_prepared.csv")
print(df.shape)
print(df.columns.tolist())

(1296675, 25)
['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'hour_of_day', 'day_of_week', 'month']


In [4]:
features = ['amt', 'category', 'gender', 'city_pop', 'hour_of_day', 'day_of_week', 'month', 'lat', 'long', 'merch_lat', 'merch_long']

X= df[features].copy()
y = df['is_fraud'].copy()

# One hot encoding for categorical features
cat_cols = ['category' ,'gender', 'day_of_week', 'month']
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"Final Features: {X.columns.tolist()}")

Final Features: ['amt', 'city_pop', 'hour_of_day', 'lat', 'long', 'merch_lat', 'merch_long', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel', 'gender_M', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Fraud rate - Train: {y_train.mean():.4%} | Test: {y_test.mean():.4%}")

Train: (1037340, 38) | Test: (259335, 38)
Fraud rate - Train: 0.5789% | Test: 0.5788%


In [ ]:
# === MANUAL DECISION TREE (Educational Implementation) ===

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=10):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
    
    def _gini(self, y):
        p = np.bincount(y) / len(y)
        return 1 - np.sum(p**2)
    
    def _best_split(self, X, y):
        best_gini = float('inf')
        best_feature, best_threshold = None, None
        n_features = X.shape[1]
        
        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds[::10]:  # sample every 10th for speed
                left_idx = X[:, feature] <= threshold
                right_idx = ~left_idx
                if len(y[left_idx]) < self.min_samples_split or len(y[right_idx]) < self.min_samples_split:
                    continue
                gini = (len(y[left_idx]) * self._gini(y[left_idx]) + 
                        len(y[right_idx]) * self._gini(y[right_idx])) / len(y)
                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = threshold
        return best_feature, best_threshold
    
    def _build_tree(self, X, y, depth=0):
        if depth >= self.max_depth or len(np.unique(y)) == 1 or len(y) < self.min_samples_split:
            return Node(value=np.bincount(y).argmax())
        
        feature, threshold = self._best_split(X, y)
        if feature is None:
            return Node(value=np.bincount(y).argmax())
        
        left_idx = X[:, feature] <= threshold
        right_idx = ~left_idx
        
        left = self._build_tree(X[left_idx], y[left_idx], depth+1)
        right = self._build_tree(X[right_idx], y[right_idx], depth+1)
        
        return Node(feature, threshold, left, right)
    
    def fit(self, X, y):
        self.tree = self._build_tree(X.values if isinstance(X, pd.DataFrame) else X, y.values)
    
    def _predict_single(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_single(x, node.left)
        return self._predict_single(x, node.right)
    
    def predict(self, X):
        return np.array([self._predict_single(x, self.tree) for x in X.values])

In [ ]:
# === MANUAL RANDOM FOREST (Bagging of Decision Trees) ===

class RandomForest:
    def __init__(self, n_trees=10, max_depth=5, max_samples=0.8):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.max_samples = max_samples
        self.trees = []
    
    def fit(self, X, y):
        n_samples = int(len(X) * self.max_samples)
        for _ in range(self.n_trees):
            # Bootstrap sampling
            idx = np.random.choice(len(X), n_samples, replace=True)
            X_boot = X.iloc[idx]
            y_boot = y.iloc[idx]
            
            tree = DecisionTree(max_depth=self.max_depth)
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
    
    def predict(self, X):
        predictions = np.zeros((len(X), self.n_trees))
        for i, tree in enumerate(self.trees):
            predictions[:, i] = tree.predict(X)
        return np.round(np.mean(predictions, axis=1)).astype(int)

In [ ]:
# Use a subsample for manual models (they are slow on 1.3M rows)
sample_idx = np.random.choice(len(X_train), 50_000, replace=False)
X_train_sample = X_train.iloc[sample_idx]
y_train_sample = y_train.iloc[sample_idx]

print("Training Manual Decision Tree...")
dt = DecisionTree(max_depth=6)
dt.fit(X_train_sample, y_train_sample)
dt_pred = dt.predict(X_test)

print("Training Manual Random Forest...")
rf = RandomForest(n_trees=15, max_depth=6)
rf.fit(X_train_sample, y_train_sample)
rf_pred = rf.predict(X_test)

print("\n=== Manual Decision Tree Results ===")
print(classification_report(y_test, dt_pred))

print("\n=== Manual Random Forest Results ===")
print(classification_report(y_test, rf_pred))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Sklearn baselines (for comparison)
sklearn_dt = DecisionTreeClassifier(max_depth=6, random_state=42)
sklearn_rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight='balanced')

sklearn_dt.fit(X_train, y_train)
sklearn_rf.fit(X_train, y_train)

print("Sklearn Decision Tree:", classification_report(y_test, sklearn_dt.predict(X_test)))
print("Sklearn Random Forest:", classification_report(y_test, sklearn_rf.predict(X_test)))